# Week 1: 환경 세팅 + 데이터 준비 + Hidden States 추출 검증

**목표**
1. 환경 세팅 및 LLaMA-3.1 8B 로드 확인
2. TruthfulQA / HaluEval / PopQA 데이터 로드 및 형식 확인
3. Hidden States 추출 파이프라인 소규모 검증 (shape 확인)
4. 캐시 저장 확인

**실행 환경**: Google Colab Pro (A100 40GB)

---

## 0. 환경 세팅

In [ ]:
# GPU 확인
!nvidia-smi | head -20

In [ ]:
# Google Drive 마운트 (체크포인트 영구 저장용)
# FinalProject.ipynb에서 %run으로 실행 시 이미 마운트됨 → "already mounted" 메시지 정상
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/self-correcting-llm'
os.makedirs(DRIVE_DIR, exist_ok=True)

# HS_CACHE: FinalProject.ipynb가 주입한 Drive 경로 우선, 없으면 Drive 루트 fallback
HS_CACHE = os.environ.get('HS_CACHE', f'{DRIVE_DIR}/hs_cache')
os.makedirs(HS_CACHE, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')
print(f'HS cache : {HS_CACHE}')

In [ ]:
# 레포 클론 및 이동 (중복 실행 안전)
import os

REPO = 'self-correcting-llm'

if os.path.exists('config.py'):
    # 이미 레포 루트 (FinalProject.ipynb에서 %run 등으로 진입한 경우)
    print(f'레포 루트 확인: {os.getcwd()}')
elif os.path.exists(REPO):
    # 레포가 이미 클론돼 있음
    %cd {REPO}
else:
    # 최초 클론
    !git clone -b phase/1-data-hidden-states \
        https://github.com/uuyeong/self-correcting-llm.git
    %cd {REPO}

In [ ]:
# 의존성 설치
!pip install -q -r requirements.txt
print('설치 완료')

In [ ]:
# HuggingFace 로그인 — Colab Secrets 사용 (권장)
# 설정 방법: Colab 왼쪽 메뉴 🔑 Secrets → 'HF_TOKEN' 이름으로 토큰 추가
from google.colab import userdata
from huggingface_hub import login

try:
    login(token=userdata.get('HF_TOKEN'))
    print('HuggingFace 로그인 성공 (Colab Secrets)')
except Exception:
    # Secrets 미설정 시 수동 입력 팝업
    login()

In [ ]:
import sys
sys.path.insert(0, '.')
import config

print(f'Primary model : {config.PRIMARY_MODEL}')
print(f'Small model   : {config.SMALL_MODEL}')
print(f'Cache dir     : {config.CACHE_DIR}')
print(f'Results dir   : {config.RESULTS_DIR}')

---
## 1. 데이터 로드 검증

### 1-A. TruthfulQA

In [ ]:
from src.data.dataset_loader import load_truthfulqa

tqa = load_truthfulqa(split='validation')

n_true = sum(r['label'] == 0 for r in tqa)
n_false = sum(r['label'] == 1 for r in tqa)
print(f'TruthfulQA 총 레코드: {len(tqa)}')
print(f'  truthful (label=0): {n_true}')
print(f'  hallucinated (label=1): {n_false}')
print()
print('샘플 레코드:')
for r in tqa[:2]:
    print(f'  Q: {r["question"]}')
    print(f'  A: {r["answer"]}')
    print(f'  label: {r["label"]}')
    print()

### 1-B. HaluEval (수동 다운로드 필요)

**다운로드 방법:**
```bash
# Colab에서 실행
!wget -q "https://github.com/RUCAIBox/HaluEval/raw/main/data/qa_data.json" -O data/halueval_qa.json
```

또는 https://github.com/RUCAIBox/HaluEval → `data/qa_data.json` 직접 다운로드 후 `data/` 폴더에 저장

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# HaluEval QA 데이터 다운로드
!wget -q "https://github.com/RUCAIBox/HaluEval/raw/main/data/qa_data.json" \
    -O data/halueval_qa.json

# config에 경로 반영 (런타임에만 적용)
config.HALUEVAL_PATH = 'data/halueval_qa.json'

# Drive에도 복사 (세션 종료 후 재사용)
!cp data/halueval_qa.json /content/drive/MyDrive/self-correcting-llm/halueval_qa.json
print('HaluEval 다운로드 완료')

In [ ]:
from src.data.dataset_loader import load_halueval

halu = load_halueval('data/halueval_qa.json')

n_true = sum(r['label'] == 0 for r in halu)
n_false = sum(r['label'] == 1 for r in halu)
print(f'HaluEval 총 레코드: {len(halu)}')
print(f'  truthful (label=0): {n_true}')
print(f'  hallucinated (label=1): {n_false}')
print()
print('샘플 레코드:')
for r in halu[:2]:
    print(f'  Q: {r["question"]}')
    print(f'  A: {r["answer"]}')
    print(f'  label: {r["label"]}')
    print()

### 1-C. PopQA

In [ ]:
from src.data.dataset_loader import load_popqa, get_domain_split

popqa = load_popqa(split='test', max_samples=2000)
domain_splits = get_domain_split(popqa)

print(f'PopQA 총 레코드: {len(popqa)}')
print('도메인 분포:')
for domain, recs in sorted(domain_splits.items()):
    print(f'  {domain}: {len(recs)}')
print()
print('샘플 레코드:')
for r in popqa[:2]:
    print(f'  Q: {r["question"]}')
    print(f'  A: {r["answer"]}')
    print(f'  domain: {r["domain"]}')
    print()

---
## 2. LLM 로드 및 메모리 확인

In [ ]:
import torch

def gpu_memory_gb():
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        return used, total
    return 0, 0

used, total = gpu_memory_gb()
print(f'GPU 메모리: {used:.1f} GB / {total:.1f} GB 사용 중')

In [ ]:
# LLaMA-3.1 8B 로드 (4-bit 양자화, ~5GB)
# HF_CACHE: FinalProject.ipynb가 HF_HOME으로 설정한 Drive 경로
# → transformers가 자동으로 HF_HOME을 캐시 디렉터리로 사용하므로 별도 지정 불필요
from src.models.llm_wrapper import LLMWrapper

print(f'모델 로드 시작: {config.PRIMARY_MODEL}')
wrapper = LLMWrapper(model_name=config.PRIMARY_MODEL, load_in_4bit=True)

used, total = gpu_memory_gb()
print(f'로드 완료 — GPU 메모리: {used:.1f} GB / {total:.1f} GB')
print(f'레이어 수 (n_layers): {wrapper.n_layers}')

In [ ]:
# 간단한 생성 테스트
test_prompt = 'Q: What is the capital of France?\nA:'
text, prompt_hs = wrapper.generate(test_prompt, max_new_tokens=30)

print(f'Prompt  : {test_prompt}')
print(f'Generated: {text}')
print(f'Hidden states layers returned: {len(prompt_hs)}')
print(f'Each HS shape: {prompt_hs[0].shape}  (expected: [1, hidden_dim])')

---
## 3. Hidden States 추출 파이프라인 검증 (소규모)

In [ ]:
import numpy as np

# TruthfulQA 에서 20개만 샘플
SAMPLE_N = 20
sample = tqa[:SAMPLE_N]
questions = [r['question'] for r in sample]
answers   = [r['answer']   for r in sample]
labels    = np.array([r['label'] for r in sample])

print(f'검증용 샘플: {SAMPLE_N}개')
print(f'  label=0 (truthful): {(labels==0).sum()}')
print(f'  label=1 (hallucinated): {(labels==1).sum()}')

In [ ]:
# Hidden States 추출 (batch_size=4)
hs = wrapper.extract_hidden_states(questions, answers, batch_size=4)

print(f'hidden_states shape: {hs.shape}')
print(f'  (n_samples={hs.shape[0]}, n_layers={hs.shape[1]}, hidden_dim={hs.shape[2]})')
print()

# 예상 shape 검증
assert hs.shape[0] == SAMPLE_N,     f'n_samples 불일치: {hs.shape[0]} vs {SAMPLE_N}'
assert hs.shape[1] == wrapper.n_layers, f'n_layers 불일치: {hs.shape[1]} vs {wrapper.n_layers}'
print('✓ Shape 검증 통과')
print(f'\n레이어별 평균 활성화 크기 (첫 5 레이어):')
for i in range(5):
    print(f'  Layer {i:2d}: mean={hs[:, i, :].mean():.4f}, std={hs[:, i, :].std():.4f}')

In [ ]:
# 마지막 레이어 기준 truthful vs hallucinated 평균 비교
last_layer_hs = hs[:, -1, :]   # (N, H)
mean_true  = last_layer_hs[labels == 0].mean(axis=0)
mean_false = last_layer_hs[labels == 1].mean(axis=0)
l2_dist = np.linalg.norm(mean_true - mean_false)

print(f'마지막 레이어 — truthful vs hallucinated 평균 L2 거리: {l2_dist:.4f}')
print('(값이 클수록 두 클래스가 hidden space에서 다르게 표상됨)')

---
## 4. 전체 TruthfulQA hidden states 추출 및 캐시 저장

> 전체 데이터(~1600개)를 추출합니다. A100 기준 약 10~15분 소요.

In [ ]:
import numpy as np
from pathlib import Path

all_questions = [r['question'] for r in tqa]
all_answers   = [r['answer']   for r in tqa]
all_labels    = np.array([r['label'] for r in tqa])

print(f'전체 추출 시작: {len(tqa)}개')
hs_full = wrapper.extract_hidden_states(all_questions, all_answers, batch_size=8)
print(f'완료: shape={hs_full.shape}')

# HS_CACHE(Drive)에 직접 저장 → 세션 종료 후에도 유지
hs_path    = Path(HS_CACHE) / 'exp1_hidden_states.npy'
label_path = Path(HS_CACHE) / 'exp1_labels.npy'
np.save(hs_path,    hs_full)
np.save(label_path, all_labels)
print(f'Drive에 저장 완료:')
print(f'  {hs_path}')
print(f'  {label_path}')

---
## 5. Week 1 완료 체크리스트

아래 셀을 실행해서 모든 항목이 ✓인지 확인 후 커밋

In [ ]:
from pathlib import Path

hs_path    = Path(HS_CACHE) / 'exp1_hidden_states.npy'
label_path = Path(HS_CACHE) / 'exp1_labels.npy'

checks = [
    ('TruthfulQA 로드',             len(tqa) > 0),
    ('HaluEval 로드',               len(halu) > 0),
    ('PopQA 로드',                  len(popqa) > 0),
    ('LLM 로드 성공',               wrapper is not None),
    ('Hidden states shape 검증',    hs_full.shape[0] == len(tqa) and hs_full.shape[1] == wrapper.n_layers),
    ('exp1_hidden_states.npy 저장', hs_path.exists()),
    ('exp1_labels.npy 저장',        label_path.exists()),
]

all_pass = True
for name, ok in checks:
    mark = 'v' if ok else 'x'
    print(f'  [{mark}] {name}')
    if not ok:
        all_pass = False

print()
if all_pass:
    print('Week 1 완료! phase/2-probing-exp1 브랜치로 이동 준비')
else:
    print('미완료 항목 있음 — 위 셀 재실행 필요')

---
## 6. Git 커밋 & 푸시 (직접 실행)

```bash
git add src/data/dataset_loader.py src/models/llm_wrapper.py \
        notebooks/week1_data_and_hidden_states.ipynb
git commit -m "feat(data): verify TruthfulQA/HaluEval/PopQA loaders

- TruthfulQA 1634개, HaluEval 10000개, PopQA 2000개 로드 확인
- hidden states shape (N, n_layers, hidden_dim) 검증 완료
- exp1_hidden_states.npy 캐시 생성 (Drive 백업)"

git push origin phase/1-data-hidden-states
```